# Compare cached and full attention outputs

Prerequisites: Causal attention, tensor dimensions and memory accounting.

In autoregressive inference, past keys and values can be reused when their token representations remain unchanged under the model's causal computation. The new query attends over cached past entries plus the new token. The companion computes one new-token output both from a full concatenated K/V array and from an incrementally extended cache, then compares the results.

For batch one, uncompressed K/V storage is 2×layers×tokens×KV heads×head dimension×bytes per element. The factor two counts keys and values. Grouped-query attention uses KV-head count here, not query-head count. This formula omits allocation overhead, temporary activations and quantization metadata; it is a capacity estimate, not a throughput benchmark.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Reuse | Past keys and values | Avoid recomputing unchanged entries |
| Append | New token K/V | Extend context |
| Check | Cached output − full output | Numerical equivalence |

## Predict

With 32 layers, 4,096 tokens, 8 KV heads, dimension 128 and two bytes per element, the raw cache is 536,870,912 bytes = 512 MiB.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import extensions as m

In [ ]:
result=m.cache()
assert result['maximum_difference']<1e-12
assert result['example_bytes']==536870912
print(result)

## Perturb and explain

Add a cache-length limit and specify its eviction policy. Compare the resulting output with full-context attention and explain why eviction changes the problem.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

The raw memory formula uses:

1. Query heads in every architecture
2. KV heads and both K and V arrays
3. Only the new token

Answer and explanations: 1: Grouped-query attention can use fewer KV heads. 2: These are the arrays actually stored. 3: Past entries dominate a long-context cache.

## Primary evidence

- [Hugging Face Transformers — Cache strategies](https://huggingface.co/docs/transformers/kv_cache)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.